---
## 1. Evaluating the Buidling Risk Agent

The Building Risk agent is intended to support city planners, insurers, and emergency-response teams with San Francisco building-level risk analysis. A demo that works once is not enough: the agent must be evaluated for whether it calls the right tools, stays grounded in the approved risk tables, handles unsupported questions gracefully, and produces useful business-facing explanations.

This notebook follows the same evaluation structure as the course Assignment 5 notebook:

| Judge type | What it checks | Why it matters for Building Risk |
|---|---|---|
| Built-in judge | Whether the answer is relevant to the user query | The response should answer the actual risk-analysis question |
| Guidelines judges | Whether the answer follows project-specific rules | The agent should cite/tool-ground its answer and stay in scope |
| Custom judge | A tailored rubric for this project | The answer should use the right Building Risk tool and explain risk drivers |
| Human review | Manual PM/AIE review of trace quality | Humans decide whether the response is acceptable for business use |

The goal is to produce at least five evaluation traces, including supported questions and graceful rejections. A separate section compares two LLMs on the same prompt for the final project requirement.


---
## 2. Install dependencies

Run this cell once per Databricks cluster/session. The notebook uses MLflow GenAI evaluation, Databricks-hosted LLM endpoints, and the agent/tooling packages from the previous notebook.


In [0]:
%pip install --upgrade "mlflow>=3.9" databricks-openai databricks-agents unitycatalog-ai[databricks] backoff pandas

dbutils.library.restartPython()

---
## 3. Verify Building Risk tables and tools

Before evaluating the agent, verify that the Building Risk tool notebook has been run and that the Unity Catalog functions exist.

Expected tools:

- `main.default.analyze_urban_fire_spread_risk`
- `main.default.analyze_emergency_access_bottlenecks`
- `main.default.rank_buildings_by_composite_risk`


In [0]:

CATALOG = "main"
SCHEMA = "default"

FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.br_sf_building_risk_features"
UC_TOOL_NAMES = [
    f"{CATALOG}.{SCHEMA}.analyze_urban_fire_spread_risk",
    f"{CATALOG}.{SCHEMA}.analyze_emergency_access_bottlenecks",
    f"{CATALOG}.{SCHEMA}.rank_buildings_by_composite_risk",
]

print("Feature table:", FEATURE_TABLE)
print("Tools:")
for tool_name in UC_TOOL_NAMES:
    print("-", tool_name)


In [0]:
# Verify the feature table exists and has rows.
try:
    feature_df = spark.table(FEATURE_TABLE)
    row_count = feature_df.count()
    print(f"Feature table row count: {row_count:,}")
    display(feature_df.limit(5))
except Exception as e:
    raise RuntimeError(
        "Could not read the Building Risk feature table. "
        "Run Building_Risk_Tool_Notebook.ipynb first, "
        "or update CATALOG/SCHEMA above."
    ) from e

# Verify all UC functions exist.
for tool_name in UC_TOOL_NAMES:
    try:
        spark.sql(f"DESCRIBE FUNCTION EXTENDED {tool_name}").display()
        print(f"OK: {tool_name}")
    except Exception as e:
        raise RuntimeError(
            f"Could not describe {tool_name}. Run the Building Risk tool notebook first."
        ) from e


---
## 4. Load the Building Risk agent

This section imports `agent.py` from the previous Building Risk agent notebook. Make sure the agent notebook has been run in the same Databricks workspace directory so that `agent.py` exists.


In [0]:
import importlib
import mlflow
import os
import sys

EXPERIMENT_NAME = "/Users/" + spark.sql("SELECT current_user()").first()[0] + "/building_risk_agent_eval"
mlflow.set_experiment(EXPERIMENT_NAME)

try:
    import agent
    importlib.reload(agent)
except Exception as e:
    raise RuntimeError(
        "Could not import agent.py. Run Building_Risk_Agent_Notebook.ipynb first "
        "or place agent.py in the same directory as this notebook."
    ) from e

AGENT = agent.AGENT
print("Loaded Building Risk agent.")
print("LLM endpoint:", agent.LLM_ENDPOINT_NAME)
print("Tools loaded by agent:")
for tool in agent.TOOL_INFOS:
    print("-", tool.name)


---
## 5. Helper functions for readable outputs

The Building Risk agent is an MLflow `ResponsesAgent`, so the output can contain structured response items. These helpers convert the response into a readable string and wrap the agent for MLflow evaluation.


In [0]:
def response_to_text(resp):
    """Convert an MLflow ResponsesAgent response to a readable string."""
    text_parts = []

    output_items = getattr(resp, "output", resp)
    for item in output_items:
        content = getattr(item, "content", None)
        if content is None and isinstance(item, dict):
            content = item.get("content")

        if content is None:
            continue

        parts = content if isinstance(content, list) else [content]
        for part in parts:
            if isinstance(part, dict) and "text" in part:
                text_parts.append(part["text"])
            elif hasattr(part, "text"):
                text_parts.append(part.text)
            elif isinstance(part, str):
                text_parts.append(part)

    if text_parts:
        newline = chr(10)
        return newline.join(text_parts)

    return str(output_items)[:1000]


def ask_agent(question, session_id="building-risk-eval-smoke-test"):
    """Run a single question through the loaded agent and print the response."""
    resp = AGENT.predict({
        "input": [{"role": "user", "content": question}],
        "custom_inputs": {"session_id": session_id},
    })
    answer = response_to_text(resp)
    print(answer)
    return answer

In [0]:

# Smoke test: this should call the composite-risk ranking tool.
ask_agent(
    "Rank the top 3 highest-risk San Francisco buildings using the composite risk score.",
    session_id="building-risk-eval-smoke-test",
)


---
## 6. Create an evaluation dataset

Each row is one evaluation trace. The `inputs` key must match the `predict_fn` argument name used later in the notebook.

This dataset includes:

- Three supported risk-analysis questions
- Two irrelevant/out-of-scope questions that the agent should reject gracefully
- One extra supported question for explanation quality

The rubric requires five examples; this notebook includes six so there is a little extra coverage.


In [0]:

manual_eval_data = [
    {
        "inputs": {
            "question": "Rank the top 5 highest-risk San Francisco buildings using the composite risk score."
        },
        "expectations": {
            "expected_tool": "rank_buildings_by_composite_risk",
            "expected_behavior": "Return a ranked list grounded in the composite risk feature table."
        },
    },
    {
        "inputs": {
            "question": "Which San Francisco buildings have the highest urban fire spread risk?"
        },
        "expectations": {
            "expected_tool": "analyze_urban_fire_spread_risk",
            "expected_behavior": "Use nearby-building density and explain cascading fire-spread risk."
        },
    },
    {
        "inputs": {
            "question": "Identify the top emergency access bottlenecks caused by dense building spacing."
        },
        "expectations": {
            "expected_tool": "analyze_emergency_access_bottlenecks",
            "expected_behavior": "Use constrained neighbor counts and explain access constraints."
        },
    },
    {
        "inputs": {
            "question": "Explain why the top composite-risk buildings are considered risky."
        },
        "expectations": {
            "expected_tool": "rank_buildings_by_composite_risk",
            "expected_behavior": "Explain risk drivers such as fire spread, access bottlenecks, and consequence indicators."
        },
    },
    {
        "inputs": {
            "question": "What is the best seafood restaurant near Fisherman's Wharf?"
        },
        "expectations": {
            "expected_tool": "none",
            "expected_behavior": "Gracefully reject because restaurant recommendations are outside the Building Risk scope."
        },
    },
    {
        "inputs": {
            "question": "Who will win the next presidential election?"
        },
        "expectations": {
            "expected_tool": "none",
            "expected_behavior": "Gracefully reject because politics/current-events forecasting is outside the Building Risk scope."
        },
    },
]

print(f"Manual evaluation dataset size: {len(manual_eval_data)}")
for row in manual_eval_data:
    print("-", row["inputs"]["question"])


In [0]:
# Create or update a UC-backed MLflow evaluation dataset.
uc_schema = f"{CATALOG}.{SCHEMA}"
evaluation_dataset_table_name = "building_risk_agent_eval"
full_table_name = f"{uc_schema}.{evaluation_dataset_table_name}"

if not spark.catalog.tableExists(full_table_name):
    eval_dataset = mlflow.genai.datasets.create_dataset(name=full_table_name)
else:
    eval_dataset = mlflow.genai.datasets.get_dataset(name=full_table_name)

eval_dataset = eval_dataset.merge_records(manual_eval_data)
print("Evaluation dataset:", full_table_name)
print("Records merged:", len(manual_eval_data))


---
## 7. Define predict function and built-in judge

The predict function converts a single text question into the request format expected by the Building Risk agent. The built-in judge checks whether the response is relevant to the user query.


In [0]:
from mlflow.genai.scorers import RelevanceToQuery


def predict_fn(question: str) -> str:
    """Run the Building Risk agent on one question and return a text answer."""
    response = AGENT.predict({
        "input": [{"role": "user", "content": question}],
        "custom_inputs": {"session_id": "building-risk-mlflow-eval"},
    })
    return response_to_text(response)


relevance_judge = RelevanceToQuery()


---
## 8. Guidelines judges

These judges encode project-specific quality bars. They intentionally reward answers that are grounded in the Building Risk tools and reject unsupported topics.


In [0]:
from mlflow.genai.scorers import Guidelines

tool_grounding_judge = Guidelines(
    name="tool_grounding",
    guidelines=(
        "For supported Building Risk questions, the response must be grounded in one of the approved tools: "
        "rank_buildings_by_composite_risk, analyze_urban_fire_spread_risk, or "
        "analyze_emergency_access_bottlenecks. The response should describe the relevant metric or data source. "
        "For out-of-scope questions, the response should not claim to use a tool."
    ),
)

scope_guardrail_judge = Guidelines(
    name="scope_guardrail",
    guidelines=(
        "The agent must stay within San Francisco building-level climate, emergency, and infrastructure risk analysis. "
        "It should gracefully reject unrelated topics such as restaurants, general travel, entertainment, politics, "
        "medical advice, or questions not answerable from the approved Building Risk datasets."
    ),
)

business_explanation_judge = Guidelines(
    name="business_explanation",
    guidelines=(
        "The response should be useful to a city planner, insurer, or emergency response stakeholder. "
        "It should explain why the result matters using clear language, not only raw IDs or scores."
    ),
)


---
## 9. Custom judge — Building Risk quality rubric

This custom judge checks the full project-specific rubric: correct tool choice, grounding, explanation quality, and graceful rejection behavior.


In [0]:
from mlflow.genai.judges import make_judge

risk_analysis_quality_judge = make_judge(
    name="risk_analysis_quality",
    instructions="""
You are evaluating a Building Risk Analysis agent for a final project.

The agent has three approved tools:
- rank_buildings_by_composite_risk: ranks San Francisco buildings by composite risk score.
- analyze_urban_fire_spread_risk: identifies buildings with high nearby-building density and urban fire-spread risk.
- analyze_emergency_access_bottlenecks: identifies dense spacing and likely emergency access constraints.

Evaluate the response using the inputs, outputs, and expectations.

Pass the response only if it meets these criteria:
1. For supported Building Risk questions, it uses or clearly relies on the expected tool or a reasonable approved tool.
2. It stays grounded in the returned Building Risk data and does not invent unsupported hazard details.
3. It explains the result in business-friendly language.
4. For out-of-scope questions, it gracefully refuses and redirects to supported Building Risk capabilities.

Fail the response if it gives a generic answer, hallucinates data, answers an irrelevant question instead of rejecting it, or does not address the user's request.

Inputs: {{ inputs }}
Outputs: {{ outputs }}
Expectations: {{ expectations }}
""",
    model="databricks:/databricks-gpt-oss-120b",
    feedback_value_type=bool,
)


---
## 10. Run MLflow evaluation and analyze

Run one evaluation job with all judges. In the MLflow UI, each row should appear as a trace with judge results.


In [0]:
print("Running Building Risk agent evaluation with all judges...")

all_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        relevance_judge,
        tool_grounding_judge,
        scope_guardrail_judge,
        business_explanation_judge,
        risk_analysis_quality_judge,
    ],
)

print("Evaluation complete. Open the MLflow experiment to review traces and judge results.")


---
## 11. Two-LLM comparison trace

The project rubric requires at least one trace that uses two different LLMs to compare performance. This section runs the same prompt against two Databricks-hosted endpoints and records latency and output for comparison.

Update `LLMS_TO_COMPARE` if your workspace uses different serving endpoint names.


In [0]:
import time
import pandas as pd

LLMS_TO_COMPARE = [
    "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b",
]

comparison_question = "Rank the top 5 highest-risk San Francisco buildings using the composite risk score and explain the main drivers."
comparison_results = []

for llm_name in LLMS_TO_COMPARE:
    print()
    print("=" * 80)
    print(f"LLM: {llm_name}")
    print("=" * 80)

    agent_for_llm = agent.ToolCallingAgent(llm_endpoint=llm_name, tools=agent.TOOL_INFOS)

    start = time.time()
    try:
        resp = agent_for_llm.predict({
            "input": [{"role": "user", "content": comparison_question}],
            "custom_inputs": {"session_id": f"two-llm-comparison-{llm_name}"},
        })
        output = response_to_text(resp)
        status = "success"
    except Exception as e:
        output = repr(e)
        status = "error"

    latency_seconds = round(time.time() - start, 2)
    print("Status:", status)
    print("Latency seconds:", latency_seconds)
    print("Output preview:", output[:1200])

    comparison_results.append({
        "model": llm_name,
        "question": comparison_question,
        "status": status,
        "latency_seconds": latency_seconds,
        "output": output,
    })

llm_comparison_df = pd.DataFrame(comparison_results)
display(llm_comparison_df)


---
## 12. ROI scoring for the two LLMs

Fill in the cost assumptions for your selected LLM endpoints. If exact endpoint pricing is not available in your workspace, use a clearly labeled estimate for the final project.

The example ROI logic below treats the higher-quality model as valuable only if the incremental business value from better answers exceeds its incremental cost.


In [0]:
# Units can be approximate cost per 1,000 agent runs, as long as both models use the same unit.
roi_assumptions = pd.DataFrame([
    {
        "model": "databricks-gpt-oss-120b",
        "estimated_cost_per_1000_runs": 12.00,
        "human_quality_score_0_to_1": 0.90,
        "estimated_business_value_per_1000_runs": 500.00,
    },
    {
        "model": "databricks-gpt-oss-20b",
        "estimated_cost_per_1000_runs": 4.00,
        "human_quality_score_0_to_1": 0.78,
        "estimated_business_value_per_1000_runs": 500.00,
    },
])

roi_assumptions["quality_adjusted_value"] = (
    roi_assumptions["human_quality_score_0_to_1"]
    * roi_assumptions["estimated_business_value_per_1000_runs"]
)
roi_assumptions["net_value"] = (
    roi_assumptions["quality_adjusted_value"]
    - roi_assumptions["estimated_cost_per_1000_runs"]
)
roi_assumptions["roi_ratio"] = (
    roi_assumptions["net_value"]
    / roi_assumptions["estimated_cost_per_1000_runs"]
)

display(roi_assumptions)


---
## 13. Written performance commentary draft

**Performance summary:**  
The Building Risk agent performed best on supported questions where the intent directly matched one of the three tools. It was strongest at ranking composite risk, identifying urban fire-spread risk, and describing emergency access bottlenecks using the feature table. The agent should be considered a prototype rather than production-ready because the current risk scores are based on building geometry and proximity features, not full FEMA flood, vegetation, elevation, or road-network overlays.

**Evaluation findings:**  
The most important evaluation checks were tool grounding, scope control, and business explanation quality. The agent should pass when it calls the appropriate Unity Catalog function, summarizes returned rows without inventing unsupported hazard details, and explains why the result matters to city planners, insurers, or emergency response teams. It should fail when it answers unrelated questions, produces generic statements without data, or claims unsupported precision.

**Two-LLM comparison:**  
The larger LLM should be recommended only if its improved explanation quality, tool choice, or rejection behavior creates enough business value to justify higher cost and latency. If the smaller LLM produces similar tool calls and grounded summaries, it may be better for high-volume deployment. If the larger LLM is noticeably better at nuanced explanations and safe refusals, it may be preferred for analyst-facing workflows.
